# Fase 3 — Construcción del Corpus 📚

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astorress/llm-lab/blob/main/notebooks/03_construccion_corpus.ipynb)

> Un pipeline **completo y ejecutable** de construcción de corpus: del texto crudo al dataset limpio, deduplicado, balanceado y listo para entrenar. "Data quality > data quantity" es la lección central de los LLM modernos.

## Objetivos
- Construir documentos según el **esquema canónico** del lab y validarlos.
- Limpiar, filtrar por **calidad** (heurísticas Gopher/C4) y **deduplicar** (exacto + MinHash/LSH desde cero).
- **Chunkear** semánticamente, **balancear** dominios con weighting, y hacer **topic modeling** y **clustering**.
- Persistir en **JSONL y Parquet** con splits estratificados.

## Prerrequisitos
Fase 2 (tokenización). Tiempo: ~30 min. Trabajamos sobre un mini-corpus demo generado *inline*.

## Índice
1. Anatomía de un corpus · 2. Esquema canónico · 3. Limpieza · 4. Calidad (Gopher/C4) · 5. Dedup exacta · 6. Dedup near (MinHash/LSH) · 7. Filtering por perplejidad · 8. Parsing Markdown · 9. Parsing de código · 10. PDFs · 11. Balancing/weighting · 12. Chunking semántico · 13. Topic modeling · 14. Clustering · 15. Datos sintéticos · 16. Persistencia · 17. Resumen

In [ ]:
# === Setup ===
import sys, subprocess
def pipq(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p], check=False)
for mod,pkg in [("sklearn","scikit-learn"),("datasketch","datasketch"),("jsonschema","jsonschema")]:
    try: __import__(mod)
    except ImportError: pipq(pkg)

import re, hashlib, unicodedata, json, random
from collections import Counter, defaultdict
import numpy as np
import matplotlib.pyplot as plt
try:
    import google.colab; IN_COLAB=True
except ImportError: IN_COLAB=False
SEED=42; random.seed(SEED); np.random.seed(SEED)
plt.rcParams["figure.figsize"]=(9,4); plt.rcParams["axes.grid"]=True
print("Setup OK | Colab:", IN_COLAB)

In [ ]:
# === Configuración y los 21 dominios (espejo de configs/corpus.yaml) ===
PROFILE = "small"   # small | large
DOMAIN_WEIGHTS = {
 "machine_learning":1.5,"deep_learning":1.5,"inteligencia_artificial":1.5,
 "ingenieria_electrica":1.5,"sistemas_potencia":1.5,"ciencia_datos":1.2,
 "matematicas":1.2,"estadistica":1.2,"optimizacion":1.2,"python":1.2,
 "programacion":1.0,"postgresql":1.0,"mongodb":1.0,"aws":1.0,"serverless":1.0,
 "iac":1.0,"apis":1.0,"arquitectura_software":1.0,"devops":1.0,
 "data_engineering":1.0,"sistemas_distribuidos":1.0}
ALPHA = 0.7   # exponente de sub-muestreo del weighting
print(f"{len(DOMAIN_WEIGHTS)} dominios | alpha={ALPHA}")

## 1. Anatomía de un corpus de pre-entrenamiento

Los LLM se entrenan sobre billones de tokens de web filtrada, código, libros y papers. La tendencia moderna es **filtrar agresivamente por calidad** en vez de acumular volumen:

| Corpus | Tamaño | Filtro principal |
|--------|--------|------------------|
| C4 (2019) | ~750 GB | heurísticas + dedup |
| The Pile (2020) | 825 GB | 22 fuentes curadas |
| RefinedWeb (2023) | ~5 TB | filtrado + dedup masivo |
| FineWeb-Edu (2024) | 1.3 T tok | clasificador de calidad educativa |

La lección (Chinchilla, Phi): a igual cómputo, **datos mejores > más datos**. Por eso este pipeline se centra en limpieza, calidad y deduplicación.

## 2. Esquema canónico del documento

Cada documento del corpus sigue el esquema de [`datasets/schemas/document.schema.json`](../datasets/schemas/document.schema.json). El `id` es el **sha256 del contenido normalizado** → determinista e idempotente (dedup gratis). Generamos un **mini-corpus demo** de ~100 documentos con duplicados y basura plantados a propósito.

In [ ]:
def normalize_text(t):
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def make_document(content, domain, source="sintetico", subcategory="", difficulty="intermedio",
                  title="", lang="es"):
    norm = normalize_text(content)
    doc_id = "sha256:" + hashlib.sha256(norm.encode()).hexdigest()[:32]
    return {"id":doc_id,"domain":domain,"subcategory":subcategory,"difficulty":difficulty,
            "source":source,"quality_score":0.0,"tokens":len(norm.split()),
            "title":title,"content":content,
            "metadata":{"language":lang,"has_code":"```" in content or "def " in content,
                        "has_math":"$" in content or "=" in content}}

# Plantillas por dominio para generar texto sintético variado
TEMPLATES = {
 "sistemas_potencia":["El flujo de carga resuelve las ecuaciones de potencia nodal para hallar voltajes y ángulos. La matriz de admitancia Ybus relaciona inyecciones de corriente y voltajes.",
   "La estabilidad transitoria evalúa si el sistema mantiene sincronismo tras una falla. El criterio de áreas iguales compara energía acelerante y desacelerante.",
   "Un transformador de potencia en una subestación reduce el voltaje de 230 kV a 13.8 kV. Las pérdidas se modelan en por unidad (p.u.)."],
 "machine_learning":["El descenso de gradiente minimiza la función de pérdida actualizando los pesos en la dirección opuesta al gradiente. La tasa de aprendizaje controla el tamaño del paso.",
   "La regularización L2 (ridge) penaliza pesos grandes añadiendo lambda por la norma al cuadrado, reduciendo el sobreajuste.",
   "Random forest combina árboles de decisión entrenados sobre muestras bootstrap, reduciendo la varianza mediante bagging."],
 "python":["def gradiente(f, x, h=1e-5): return (f(x+h)-f(x-h))/(2*h)  # derivada numérica central",
   "Una list comprehension `[x**2 for x in range(10)]` es más legible y rápida que un bucle for explícito.",
   "Los generadores con `yield` producen valores perezosamente, ahorrando memoria en secuencias grandes."],
 "postgresql":["SELECT domain, COUNT(*) FROM documents GROUP BY domain HAVING COUNT(*) > 10 ORDER BY 2 DESC;",
   "Un índice B-tree acelera las búsquedas por igualdad y rango en O(log n). CREATE INDEX idx ON tabla(columna).",
   "Las transacciones ACID garantizan atomicidad, consistencia, aislamiento y durabilidad mediante WAL y MVCC."],
 "aws":["Una función Lambda se ejecuta sin servidor y escala automáticamente; se factura por milisegundo de ejecución.",
   "S3 ofrece almacenamiento de objetos con 11 nueves de durabilidad. Las clases de almacenamiento optimizan costo según acceso.",
   "DynamoDB es una base NoSQL gestionada con latencia de milisegundos y escalado automático por particiones."],
 "matematicas":["La descomposición en valores singulares factoriza A = U Σ Vᵀ, donde Σ contiene los valores singulares ordenados.",
   "El gradiente de una función escalar apunta en la dirección de máximo crecimiento; su norma mide la pendiente.",
   "Una matriz es definida positiva si todos sus autovalores son positivos, lo que garantiza un mínimo único en optimización cuadrática."],
}
DEFAULT_TXT = "Este es un documento técnico del dominio {d} que cubre conceptos fundamentales y aplicaciones prácticas con vocabulario especializado."

corpus = []
for dom in DOMAIN_WEIGHTS:
    tpls = TEMPLATES.get(dom, [DEFAULT_TXT.format(d=dom)])
    n = random.randint(3, 7)
    for i in range(n):
        base = random.choice(tpls)
        content = base + f" (variante {i}, contexto adicional sobre {dom}.)"
        corpus.append(make_document(content, dom, title=f"{dom} doc {i}"))
# Plantar duplicados exactos y casi-duplicados
corpus.append(make_document(corpus[0]["content"], corpus[0]["domain"]))        # duplicado exacto
corpus.append(make_document(corpus[1]["content"]+" ", corpus[1]["domain"]))     # casi-duplicado (espacio)
corpus.append(make_document(corpus[2]["content"].replace("voltaje","tension"), corpus[2]["domain"]))  # near-dup
# Plantar basura
corpus.append(make_document("aaa!!! $$$ 123 @@@ ###", "programacion"))
corpus.append(make_document("ok", "python"))                                    # demasiado corto
print(f"Corpus demo: {len(corpus)} documentos")

# Validar contra el schema (si jsonschema está disponible)
try:
    import jsonschema
    schema_path = "../datasets/schemas/document.schema.json"
    import os
    if os.path.exists(schema_path):
        schema = json.load(open(schema_path))
        jsonschema.validate(corpus[0], schema)
        print("✓ documento valida contra el schema")
except Exception as e:
    print("validación schema omitida:", type(e).__name__)

## 3. Limpieza de texto

Normalización Unicode (NFKC), eliminación de boilerplate/HTML/URLs y colapso de espacios. La normalización ya se aplicó en `make_document`; aquí mostramos el efecto sobre texto sucio.

In [ ]:
def clean(t):
    t = unicodedata.normalize("NFKC", t)
    t = re.sub(r"<[^>]+>", " ", t)                 # HTML
    t = re.sub(r"http\S+|www\.\S+", " ", t)        # URLs
    t = re.sub(r"\S+@\S+", " ", t)                 # emails
    t = re.sub(r"\s+", " ", t).strip()
    return t
dirty = "Visita  <b>https://ejemplo.com</b>  ahora!!!   contacto@mail.com   café técnico"
print("ANTES:", repr(dirty))
print("DESPUÉS:", repr(clean(dirty)))

## 4. Heurísticas de calidad (estilo Gopher / C4)

Funciones puras componibles que puntúan cada documento en [0,1]. Filtramos los que caen bajo un umbral.

In [ ]:
def alpha_ratio(t):
    return sum(c.isalpha() or c.isspace() for c in t)/max(len(t),1)
def symbol_ratio(t):
    return sum(not c.isalnum() and not c.isspace() for c in t)/max(len(t),1)
def repeated_lines_ratio(t):
    lines=[l for l in t.split(".") if l.strip()]
    if not lines: return 0.0
    return 1 - len(set(lines))/len(lines)
def quality_score(t):
    s=1.0
    n=len(t)
    if n<50: s-=0.5
    if n>50000: s-=0.2
    if alpha_ratio(t)<0.6: s-=0.4
    if symbol_ratio(t)>0.15: s-=0.3
    s-=0.3*repeated_lines_ratio(t)
    return max(0.0,min(1.0,s))

for d in corpus: d["quality_score"]=quality_score(d["content"])
scores=[d["quality_score"] for d in corpus]
plt.hist(scores, bins=20); plt.xlabel("quality_score"); plt.ylabel("documentos")
plt.title("Distribución de calidad"); plt.axvline(0.5, color="r", ls="--", label="umbral"); plt.legend(); plt.show()
kept=[d for d in corpus if d["quality_score"]>=0.5]
print(f"{len(corpus)} -> {len(kept)} documentos tras filtro de calidad (rechazados {len(corpus)-len(kept)})")

In [ ]:
# Distribución de cada heurística de calidad sobre el corpus demo
import numpy as np
metrics={"alpha_ratio":[alpha_ratio(d["content"]) for d in corpus],
         "symbol_ratio":[symbol_ratio(d["content"]) for d in corpus],
         "longitud":[len(d["content"]) for d in corpus],
         "quality_score":[quality_score(d["content"]) for d in corpus]}
fig,axes=plt.subplots(1,4,figsize=(15,3))
for ax,(k,v) in zip(axes,metrics.items()):
    ax.hist(v,bins=20); ax.set_title(k,fontsize=9)
plt.tight_layout(); plt.show()
# Rechazos por regla (cuántos docs falla cada criterio)
reasons={"muy corto (<50)":sum(len(d["content"])<50 for d in corpus),
         "alpha<0.6":sum(alpha_ratio(d["content"])<0.6 for d in corpus),
         "símbolos>0.15":sum(symbol_ratio(d["content"])>0.15 for d in corpus)}
plt.figure(figsize=(7,3)); plt.bar(reasons.keys(),reasons.values(),color="crimson")
plt.title("Documentos que violan cada regla de calidad"); plt.tight_layout(); plt.show()

## 5. Deduplicación exacta

Como el `id` es el sha256 del contenido normalizado, los duplicados exactos comparten id: basta agrupar por id.

In [ ]:
def exact_dedup(docs):
    seen={};
    for d in docs:
        seen.setdefault(d["id"], d)
    return list(seen.values())
dd = exact_dedup(kept)
print(f"{len(kept)} -> {len(dd)} tras dedup exacta (eliminados {len(kept)-len(dd)} duplicados)")

## 6. Deduplicación near-duplicate: MinHash + LSH desde cero

Dos documentos casi iguales no comparten hash exacto. Medimos similitud con **Jaccard** sobre *shingles* (n-gramas de palabras):

$$J(A,B) = \frac{|A \cap B|}{|A \cup B|}$$

**MinHash**: con una permutación aleatoria de los shingles, $P(\text{minhash}(A)=\text{minhash}(B)) = J(A,B)$. Usando $k$ permutaciones estimamos $J$ por la fracción de minhashes que coinciden. **LSH por bandas** agrupa candidatos eficientemente.

In [ ]:
def shingles(text, n=3):
    w = normalize_text(text).split()
    return set(tuple(w[i:i+n]) for i in range(max(len(w)-n+1,1)))
def jaccard(a,b):
    return len(a&b)/max(len(a|b),1)

# MinHash desde cero: k funciones hash (a*x+b mod p)
def minhash_signature(sh, k=64, seed=SEED):
    rng=np.random.default_rng(seed); p=(1<<61)-1
    a=rng.integers(1,p,k); b=rng.integers(0,p,k)
    hs=np.array([hash(s) & 0xffffffffffffffff for s in sh], dtype=np.uint64)
    if len(hs)==0: return np.zeros(k,dtype=np.uint64)
    sig=np.empty(k,dtype=np.uint64)
    for i in range(k):
        sig[i]=np.min((a[i]*hs + b[i]) % p)
    return sig
def est_jaccard(s1,s2): return float((s1==s2).mean())

# Demostrar P(minhash igual) ≈ Jaccard sobre los near-duplicates plantados
A=shingles(corpus[2]["content"]); B=shingles(corpus[-3]["content"])  # voltaje vs tension
sigA, sigB = minhash_signature(A), minhash_signature(B)
print(f"Jaccard real = {jaccard(A,B):.3f} | MinHash estimado = {est_jaccard(sigA,sigB):.3f}")

# Curva LSH: P(par candidato) = 1-(1-s^r)^b
plt.figure(figsize=(8,4))
s=np.linspace(0,1,100)
for b,r in [(16,4),(8,8),(4,16)]:
    plt.plot(s, 1-(1-s**r)**b, label=f"b={b}, r={r}")
plt.axvline(0.85, color="gray", ls="--", label="umbral 0.85")
plt.xlabel("similitud Jaccard real"); plt.ylabel("P(par candidato)")
plt.title("Curva S de LSH por bandas"); plt.legend(); plt.show()

In [ ]:
# Near-dedup: comparar todos los pares (corpus pequeño) y eliminar similares > 0.85
def near_dedup(docs, thr=0.85):
    sigs=[minhash_signature(shingles(d["content"])) for d in docs]
    keep=[True]*len(docs)
    for i in range(len(docs)):
        if not keep[i]: continue
        for j in range(i+1,len(docs)):
            if keep[j] and est_jaccard(sigs[i],sigs[j])>thr:
                keep[j]=False
    return [d for d,k in zip(docs,keep) if k]
nd = near_dedup(dd)
print(f"{len(dd)} -> {len(nd)} tras near-dedup (eliminados {len(dd)-len(nd)})")

### 6.1 Verificación empírica: P(minhash igual) ≈ Jaccard

La propiedad que hace funcionar MinHash es que la probabilidad de que dos conjuntos compartan el mínimo de
una permutación aleatoria es exactamente su similitud de Jaccard. Lo comprobamos sobre muchos pares.

In [ ]:
# Scatter: Jaccard real vs estimación MinHash sobre pares aleatorios del corpus
import numpy as np
pares=[]
for i in range(0,min(40,len(corpus))):
    for j in range(i+1,min(40,len(corpus))):
        A=shingles(corpus[i]["content"]); B=shingles(corpus[j]["content"])
        if A and B:
            real=len(A&B)/len(A|B)
            est=est_jaccard(minhash_signature(A), minhash_signature(B))
            pares.append((real,est))
pares=np.array(pares)
plt.figure(figsize=(5,5)); plt.scatter(pares[:,0],pares[:,1],alpha=0.4,s=15)
plt.plot([0,1],[0,1],"r--",label="ideal y=x"); plt.xlabel("Jaccard real"); plt.ylabel("MinHash estimado")
plt.title("MinHash estima la similitud de Jaccard"); plt.legend(); plt.tight_layout(); plt.show()
if len(pares): print(f"Error absoluto medio de la estimación: {np.abs(pares[:,0]-pares[:,1]).mean():.3f}")

## 7. Filtering por perplejidad (conceptual)

Un LM pequeño puede puntuar la "fluidez" de un texto: perplejidad muy alta ⇒ texto incoherente o ruido. Aquí usamos un proxy heurístico (diversidad léxica) por reproducibilidad sin descargar modelos.

In [ ]:
def pseudo_perplexity(t):
    w=normalize_text(t).split()
    if len(w)<3: return 1000.0
    diversity=len(set(w))/len(w)          # proxy: repetición excesiva => baja diversidad
    return 1.0/max(diversity,0.01)
ppx=[pseudo_perplexity(d["content"]) for d in nd]
plt.hist(ppx, bins=20); plt.xlabel("pseudo-perplejidad"); plt.ylabel("docs")
plt.title("Filtrado por fluidez (proxy)"); plt.show()
print("Nota: en producción se usa un LM real (KenLM o un GPT pequeño) — ver notebook 08.")

## 8. Parsing de Markdown

Extraemos estructura: headers → subcategoría, bloques de código → `has_code`, fórmulas → `has_math`.

In [ ]:
md_doc = '''# Flujo de Carga
El metodo de Newton-Raphson resuelve $f(x)=0$ iterando.
```python
def newton(f, df, x0):
    return x0 - f(x0)/df(x0)
```
## Convergencia
Cuadratica cerca de la raiz.'''
def parse_markdown(t):
    headers=re.findall(r"^#+\s+(.+)$", t, re.M)
    code_blocks=re.findall(r"```(\w*)\n(.*?)```", t, re.S)
    has_math=bool(re.search(r"\$.+?\$", t))
    return {"headers":headers,"n_code_blocks":len(code_blocks),"has_math":has_math,
            "subcategory":headers[0] if headers else ""}
print(parse_markdown(md_doc))

## 9. Parsing de código fuente

El código mejora el razonamiento de los LLM (estructura lógica explícita). Separamos docstrings, comentarios y código con `ast` y `tokenize`.

In [ ]:
import ast, io, tokenize
src = '''def potencia_aparente(v, i):
    \"\"\"Calcula S = V * I* (potencia aparente compleja).\"\"\"
    # conjugado de la corriente
    return v * i.conjugate()'''
tree = ast.parse(src)
docstring = ast.get_docstring(tree.body[0])
comments=[tok.string for tok in tokenize.generate_tokens(io.StringIO(src).readline) if tok.type==tokenize.COMMENT]
print("docstring:", docstring)
print("comentarios:", comments)
print("funciones:", [n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)])

## 10. Extracción desde PDFs

Los libros y papers técnicos vienen en PDF. Los retos reales: columnas múltiples, tablas, ecuaciones (se pierden), ligaduras tipográficas. Herramientas:

| Herramienta | Fuerte en | Débil en |
|-------------|-----------|----------|
| pypdf | texto simple | layout, tablas, math |
| GROBID | papers científicos (estructura) | requiere servidor |
| marker / nougat | math y tablas (ML) | lento, GPU |

```python
# Esquema típico (no ejecutado aquí):
from pypdf import PdfReader
text = "\n".join(p.extract_text() for p in PdfReader("libro.pdf").pages)
```
Para math, **nougat** (Meta) convierte PDF científico a Markdown/LaTeX.

## 11. Balancing y weighting de dominios

Sin control, los dominios con más texto dominan. Aplicamos muestreo sub-lineal:

$$P(d) \propto w_d \cdot n_d^{\alpha}, \quad \alpha=0.7$$

El exponente $\alpha<1$ sobre-muestrea dominios pequeños y sub-muestrea los grandes (como en Llama/XLM-R).

In [ ]:
tokens_por_dom=defaultdict(int)
for d in nd: tokens_por_dom[d["domain"]]+=d["tokens"]
doms=list(DOMAIN_WEIGHTS)
n=np.array([tokens_por_dom.get(d,1) for d in doms], dtype=float)
w=np.array([DOMAIN_WEIGHTS[d] for d in doms])
raw = n/n.sum()
probs = w*(n**ALPHA); probs/=probs.sum()
x=np.arange(len(doms))
plt.figure(figsize=(13,4)); plt.bar(x-0.2, raw, 0.4, label="proporción cruda (tokens)")
plt.bar(x+0.2, probs, 0.4, label=f"con weighting (α={ALPHA})")
plt.xticks(x, doms, rotation=90, fontsize=7); plt.ylabel("prob. de muestreo")
plt.title("Balanceo de dominios"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# WeightedSampler: muestrear según P(d) y verificar que la distribución empírica coincide
import numpy as np
rng=np.random.default_rng(0)
dom_list=list(DOMAIN_WEIGHTS); p=np.array([probs[i] if False else 0 for i in range(len(dom_list))])
# usar las probs calculadas arriba (vector 'probs' de la celda anterior)
p=np.array(probs); p=p/p.sum()
samples=rng.choice(len(dom_list), size=5000, p=p)
emp=np.bincount(samples, minlength=len(dom_list))/5000
x=np.arange(len(dom_list))
plt.figure(figsize=(13,4)); plt.bar(x-0.2,p,0.4,label="P(d) teórica")
plt.bar(x+0.2,emp,0.4,label="frecuencia empírica (5000 muestras)")
plt.xticks(x,dom_list,rotation=90,fontsize=7); plt.ylabel("prob"); plt.legend()
plt.title("WeightedSampler: el muestreo reproduce la distribución de weighting"); plt.tight_layout(); plt.show()

## 12. Chunking semántico

Para entrenar con contexto fijo hay que partir documentos largos. Mejor que cortar cada N tokens: detectar **fronteras de tópico** por caídas de similitud coseno entre ventanas consecutivas (TF-IDF).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
long_doc = " ".join([TEMPLATES["sistemas_potencia"][0]]*2 + [TEMPLATES["machine_learning"][0]]*2 + [TEMPLATES["python"][1]]*2)
sentences=[s.strip() for s in long_doc.split(".") if s.strip()]
if len(sentences)>=3:
    vec=TfidfVectorizer().fit_transform(sentences).toarray()
    sims=[float(np.dot(vec[i],vec[i+1])/(np.linalg.norm(vec[i])*np.linalg.norm(vec[i+1])+1e-9)) for i in range(len(vec)-1)]
    plt.figure(figsize=(9,3)); plt.plot(sims,"o-"); plt.axhline(np.mean(sims)-np.std(sims),color="r",ls="--",label="umbral de corte")
    plt.xlabel("frontera entre oraciones"); plt.ylabel("similitud coseno"); plt.title("Detección de cambios de tópico"); plt.legend(); plt.show()
    print("Valles = fronteras semánticas → puntos de corte para chunks coherentes")

## 13. Topic modeling (LDA)

LDA descubre tópicos latentes (distribuciones de palabras) sin supervisión. Útil para auditar la composición temática del corpus.

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
texts=[d["content"] for d in nd]
cv=CountVectorizer(max_features=80, stop_words=None)
X=cv.fit_transform(texts)
lda=LatentDirichletAllocation(n_components=5, random_state=SEED).fit(X)
words=cv.get_feature_names_out()
for k,comp in enumerate(lda.components_):
    top=[words[i] for i in comp.argsort()[-6:][::-1]]
    print(f"Tópico {k}: {', '.join(top)}")

## 14. Clustering semántico

Agrupamos documentos por contenido (TF-IDF + KMeans) y comparamos los clusters con los dominios reales mediante PCA 2D.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=100).fit_transform(texts).toarray()
k=min(6, len(texts)//3)
km=KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(tfidf)
coords=PCA(n_components=2, random_state=SEED).fit_transform(tfidf)
fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].scatter(coords[:,0],coords[:,1],c=km.labels_,cmap="tab10",s=40); axes[0].set_title("Coloreado por cluster KMeans")
dom_ids=[list(DOMAIN_WEIGHTS).index(d["domain"]) for d in nd]
sc=axes[1].scatter(coords[:,0],coords[:,1],c=dom_ids,cmap="tab20",s=40); axes[1].set_title("Coloreado por dominio real")
plt.tight_layout(); plt.show()

## 15. Datos sintéticos (conceptual)

Cuando el conocimiento está en libros no reutilizables, se generan Q&A sintéticos *a partir* del conocimiento (no copia textual). Taxonomía: **self-instruct**, **destilación** (de un modelo grande), **textbooks** (Phi), **evol-instruct**. Riesgo: *model collapse* si se entrena recursivamente sobre datos sintéticos.

In [ ]:
SYNTH_PROMPT = '''Eres un experto en {dominio}. Genera una pregunta tecnica de dificultad {dificultad}
y su respuesta correcta y autocontenida. Formato JSON: {{"pregunta": "...", "respuesta": "..."}}.
Tema: {tema}.'''
print(SYNTH_PROMPT.format(dominio="sistemas de potencia", dificultad="avanzado", tema="estabilidad transitoria"))
# Ejemplo de salida esperada (generada por un LLM grande en producción):
print(json.dumps({"pregunta":"¿Qué establece el criterio de áreas iguales?",
                  "respuesta":"Que el sistema es transitoriamente estable si el área desacelerante "
                  "disponible es mayor o igual al área acelerante durante la falla."}, ensure_ascii=False, indent=2))

## 16. Persistencia: JSONL + Parquet + splits

Guardamos el corpus final en JSONL (intercambio) y Parquet (analítico, ~5-10× menor). Hacemos splits 95/4/1 **estratificados por dominio**.

In [ ]:
import os
out_dir = "/content/corpus" if IN_COLAB else "/tmp/llm_lab_corpus"
os.makedirs(out_dir, exist_ok=True)
# JSONL
jsonl_path=os.path.join(out_dir,"corpus.jsonl")
with open(jsonl_path,"w",encoding="utf-8") as f:
    for d in nd: f.write(json.dumps(d, ensure_ascii=False)+"\n")
sizes={"jsonl":os.path.getsize(jsonl_path)}
# Parquet (si pyarrow disponible)
try:
    import pandas as pd
    df=pd.json_normalize(nd)
    pq=os.path.join(out_dir,"corpus.parquet"); df.to_parquet(pq)
    sizes["parquet"]=os.path.getsize(pq)
except Exception as e:
    print("Parquet omitido:", type(e).__name__)
plt.bar(sizes.keys(), sizes.values()); plt.ylabel("bytes"); plt.title("Tamaño en disco por formato"); plt.show()

# Splits estratificados por dominio
random.shuffle(nd)
by_dom=defaultdict(list)
for d in nd: by_dom[d["domain"]].append(d)
train,val,test=[],[],[]
for dom,docs in by_dom.items():
    n=len(docs); ntr=max(1,int(0.95*n)); nva=int(0.04*n)
    train+=docs[:ntr]; val+=docs[ntr:ntr+nva]; test+=docs[ntr+nva:]
print(f"splits → train={len(train)} val={len(val)} test={len(test)}")
print(f"corpus final escrito en {out_dir}")

## 16.1 Estadísticas finales del corpus

Un resumen cuantitativo del corpus resultante: tamaño, cobertura por dominio y distribución de longitudes —
el tipo de "ficha técnica" que acompaña a cualquier dataset serio (datasheet).

In [ ]:
# Ficha técnica del corpus demo procesado
import numpy as np
lens=[len(d["content"].split()) for d in nd]
dom_counts={}
for d in nd: dom_counts[d["domain"]]=dom_counts.get(d["domain"],0)+1
print(f"Documentos finales: {len(nd)}")
print(f"Palabras totales (aprox tokens): {sum(lens):,}")
print(f"Longitud por doc: media={np.mean(lens):.0f}, mediana={np.median(lens):.0f}, max={max(lens)}")
print(f"Dominios cubiertos: {len(dom_counts)}/21")
fig,axes=plt.subplots(1,2,figsize=(14,4))
axes[0].bar(range(len(dom_counts)), sorted(dom_counts.values(), reverse=True))
axes[0].set_title("Documentos por dominio (ordenado)"); axes[0].set_xlabel("dominio (rank)")
axes[1].hist(lens,bins=25); axes[1].set_title("Distribución de longitudes (palabras/doc)")
axes[1].set_xlabel("palabras"); plt.tight_layout(); plt.show()
print("\\nEste mismo pipeline, sobre los documentos semilla reales del repo, se ejecuta con")
print("`python scripts/build_corpus.py` y produce datasets/processed/ + splits/ (ver datasets/README.md).")

## 17. Resumen, ejercicios y referencias

### Resumen
Construimos un pipeline completo: esquema → limpieza → calidad → dedup exacta y near (MinHash/LSH desde cero) → parsing (MD, código, PDF) → balanceo con weighting → chunking semántico → topic modeling → clustering → persistencia con splits estratificados. El código modular está en [`src/llm_lab/data/`](../src/llm_lab/data/).

### Ejercicios
1. **(★)** Ajusta el umbral de near-dedup a 0.7 y mide cuántos documentos extra se eliminan.
2. **(★★)** Implementa LSH por bandas real (con buckets) y compara su velocidad con la comparación todos-contra-todos.
3. **(★★)** Sustituye el proxy de perplejidad por un GPT-2 pequeño de HF y re-filtra.
4. **(★★★)** Implementa dedup *entre* splits (ningún near-dup de test en train) y verifica que no hay fuga.

### Referencias
- Raffel et al. (2020). *Exploring the Limits of Transfer Learning (C4/T5)*. [arXiv:1910.10683](https://arxiv.org/abs/1910.10683)
- Gao et al. (2020). *The Pile*. [arXiv:2101.00027](https://arxiv.org/abs/2101.00027)
- Penedo et al. (2023). *RefinedWeb*. [arXiv:2306.01116](https://arxiv.org/abs/2306.01116)
- Lee et al. (2022). *Deduplicating Training Data Makes Language Models Better*. [arXiv:2107.06499](https://arxiv.org/abs/2107.06499)
- Rae et al. (2021). *Gopher* (heurísticas de calidad). [arXiv:2112.11446](https://arxiv.org/abs/2112.11446)